# Analisis de `suma_matrices.json`

In [ ]:
# Figuras VECTORIALES en vez de PNG. Tres motivos, en orden de importancia:
#   · git guarda cada blob comprimido, y un SVG de líneas comprime 5 veces
#     mejor que el mismo gráfico en PNG (150 KB contra 761 KB este notebook);
#   · se puede ampliar sin que se pixele;
#   · apagando la fecha que matplotlib incrusta en el SVG, dos ejecuciones
#     seguidas con los mismos datos dan bytes IDÉNTICOS — así reejecutar el
#     notebook no crea una copia nueva en la historia del repositorio.
%config InlineBackend.figure_formats = ['svg']
%config InlineBackend.print_figure_kwargs = {'metadata': {'Date': None}}

import json
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator

plt.rcParams["svg.hashsalt"] = "modulo05"   # ids estables entre corridas
plt.rcParams["figure.figsize"] = (10, 5.5)
plt.rcParams["axes.grid"] = True

# ── Ayudantes comunes a todas las secciones ─────────────────────────────────
#

print("Ayudantes cargados")

## 1. Visualización de Matrices mediante Mapas de Calor (Heatmaps)

Al trabajar con estructuras de datos bidimensionales (\(M \times N\)), la representación lineal oculta la relación espacial entre filas y columnas. 

En esta sección, utilizamos `plt.imshow()` para reconstruir la topografía exacta de las matrices procesadas por la GPU. Dado que el archivo JSON exporta los vectores de forma plana (linealizada en Row-Major Layout), empleamos `.reshape(M, N)` de NumPy para devolverles su geometría original.

Analizaremos tres mapas de calor en paralelo:
* **Matriz A:** Distribución uniforme (ruido plano).
* **Matriz B:** Distribución normal (textura concéntrica difusa).
* **Matriz C:** El resultado de la suma ejecutada de forma concurrente en la GPU.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Cargar el JSON crudo para extraer metadatos y vectores independientes
with open('./build/suma_matrices.json', 'r') as f:
    datos_crudos = json.load(f)

# Extraer dimensiones de la matriz
M = datos_crudos['M']
N = datos_crudos['N']

# 2. Reconstruir las matrices bidimensionales originales usando reshape
matriz_A = np.array(datos_crudos['A']).reshape(M, N)
matriz_B = np.array(datos_crudos['B']).reshape(M, N)
matriz_C = np.array(datos_crudos['C']).reshape(M, N)

# 3. Graficar los mapas de calor en paralelo (Subplots)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Mapa de Calor - Matriz A (Uniforme)
imA = axes[0].imshow(matriz_A, cmap='viridis', aspect='auto')
axes[0].set_title(f'Matriz A (Uniforme [{datos_crudos["a_min"]}, {datos_crudos["a_max"]}])')
fig.colorbar(imA, ax=axes[0], label='Amplitud')

# Mapa de Calor - Matriz B (Normal)
imB = axes[1].imshow(matriz_B, cmap='plasma', aspect='auto')
axes[1].set_title(f'Matriz B (Normal $\\mu={datos_crudos["b_mu"]}, \\sigma={datos_crudos["b_sigma"]}$)')
fig.colorbar(imB, ax=axes[1], label='Amplitud')

# Mapa de Calor - Matriz C (Resultado Suma GPU)
imC = axes[2].imshow(matriz_C, cmap='magma', aspect='auto')
axes[2].set_title('Matriz C Resultado (A + B)')
fig.colorbar(imC, ax=axes[2], label='Amplitud')

# Ajustes estéticos de los ejes para que muestren las dimensiones de la matriz
for ax in axes:
    ax.set_xlabel('Columnas ($N$)')
    ax.set_ylabel('Filas ($M$)')
    ax.set_xticks(np.arange(0, N, 4))
    ax.set_yticks(np.arange(0, M, 2))
    ax.grid(False) # Apagamos la grilla de líneas tradicional para no ensuciar el mapa de calor

plt.suptitle('Análisis Espacial de Matrices Concurrentes mediante Mapas de Calor en CUDA', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()


## 2. Validación de Convolución y Consistencia Estadística

Para asegurar que el mapeo bidimensional de los hilos en el espacio gráfico (`blockDim`, `threadIdx`, etc.) no corrompió el orden de las celdas (Row-Major), realizamos un análisis estadístico clásico agregando los datos de forma global. 

La matriz resultante \(C\) debe cumplir estrictamente con los criterios de linealidad de la esperanza y adición de varianzas independientes de manera global.

In [ ]:
# 1. Aplanar las matrices estables para el cálculo estadístico global
flat_A = matriz_A.flatten()
flat_B = matriz_B.flatten()
flat_C = matriz_C.flatten()

# 2. Cálculos estadísticos medidos
mu_a, var_a = flat_A.mean(), flat_A.var()
mu_b, var_b = flat_B.mean(), flat_B.var()
mu_c, var_c = flat_C.mean(), flat_C.var()

# 3. Cálculos teóricos basados en las variables reales del JSON (Corregido)
media_teorica_c = ((datos_crudos['a_min'] + datos_crudos['a_max']) / 2.0) + datos_crudos['b_mu']
varianza_teorica_a = ((datos_crudos['a_max'] - datos_crudos['a_min']) ** 2) / 12.0
varianza_teorica_b = datos_crudos['b_sigma'] ** 2
varianza_teorica_c = varianza_teorica_a + varianza_teorica_b

# Imprimir el reporte dinámico
print(f"--- Reporte de Consistencia de Estructura de Datos (M={M}, N={N}) ---")
print(f"Media global de C:    Esperada: {media_teorica_c:.4f}  | Medida Real: {mu_c:.4f}")
print(f"Varianza global de A: Esperada: {varianza_teorica_a:.4f}  | Medida Real: {var_a:.4f}")
print(f"Varianza global de B: Esperada: {varianza_teorica_b:.4f}  | Medida Real: {var_b:.4f}")
print(f"Varianza global de C: Esperada: {varianza_teorica_c:.4f}  | Medida Real: {var_c:.4f}")
